In [0]:
import pyspark.sql.functions as f
from pyspark.sql.functions import lit, col, explode, array, array_contains, array_remove, array_distinct, array_union, array_except, array_intersect, array_contains, array_remove, array_sort, array_max, array_min, array_contains, array_remove, array_distinct, array_union, array_except, array_intersect, array
import json

In [0]:
g_env = 'DEV'
g_ucBronze = 'dev_hub_bronze'
v_p_srcSchema = 'lh_ax_idr'

In [0]:
df_tables = spark.sql("""
    SELECT table_catalog, table_schema, table_name
    FROM dev_hub_silver.information_schema.tables
    WHERE table_catalog = 'dev_hub_silver'
    AND
    table_schema = 'lh_ax_idr'
""")

tables = []
for i in df_tables.collect():
    tables.append(i[2])
print(tables)

##extract using etteration

In [0]:
df_meta = spark.table(f'dev_bronze.poc._meta')
pk_dict = {}
for tbl in tables:
    pk_list = df_meta.filter(f.upper(df_meta.TABLE_NM) == tbl.upper()).select("PK_COL_NM").toPandas()["PK_COL_NM"].tolist()
    pk_dict[tbl] = pk_list

pk_json = json.dumps(pk_dict)
print(pk_json)
pk_json_nested = json.dumps({"keys": list(pk_dict.keys()), "value": pk_dict})
dbutils.jobs.taskValues.set(key="pk_json", value=pk_json_nested)

In [0]:
display(list(pk_dict.keys()))

## extract using join

In [0]:
meta_join = df_meta.join(df_tables, f.upper(df_meta.TABLE_NM) == f.upper(df_tables.table_name), 'left')
missing_tables = df_tables.filter(~f.upper(df_tables.table_name).isin([x.TABLE_NM.upper() for x in df_meta.collect()]))
_meta_agg_pk = df_meta.groupBy("TABLE_NM").pivot("PK_COL_NM").agg(f.lit(1)).fillna(0)
display(_meta_agg_pk)

In [0]:
ls_meta = df_meta.groupby("schema_nm","table_nm")\
        .agg(f.collect_list('PK_COL_NM').alias('pk'))\
        .collect()
# ls_meta = df_meta.collect()

ls_tbl = []
for i in ls_meta:
#     ls_tbl.append([i[0],v_p_srcSchema, i[1],i[2]])        
    ls_tbl.append([v_p_srcSchema, v_p_srcSchema, i[1], i[2]])

In [0]:

for i in pk_dict:
    v_src_schema = i[0]
    v_p_srcSchema = i[1]
    v_tgt_tbl = i[2]
    v_pk = i[3]
    print( v_tgt_tbl, v_pk)  

In [0]:
print(pk_dict)
for tbl, pk_list in pk_dict.items():
    print(f"Table: {tbl}")
    print(f"Primary Keys: {pk_list}")
    print("-----------------")

In [0]:
sdf = spark.table(f'{g_ucBronze}.{v_p_srcSchema}.{v_tgt_tbl}')
sdf.createOrReplaceTempView('sdf')

In [0]:
for i in pk_dict:
    tbl = i[2]
    pk = i[3]
    pk_ls = ', '.join(pk)
    print(tbl)
    print("-----------------")
    print(pk_ls)
    print("\n")    

In [0]:
print(pk_ls)

In [0]:
def sync_idr_tables(pk_json):
    for i in ls_tbl:
        tbl = i[2]
        pk = i[3]
        pk_ls = ', '.join(pk)
        print(tbl)
        print("-----------------")
        print(pk_ls)
        print("\n") 


    # Deduplicate source
    deduped = spark.sql(f'''
        SELECT *
        FROM (
            SELECT *,
                   ROW_NUMBER() OVER (PARTITION BY {pk}
                                      ORDER BY data_received_utc_dttm DESC) AS rn
            FROM {sdf}
        ) t
        WHERE rn = 1
    ''')
    final = deduped.drop("rn")
    final.createOrReplaceTempView("final")

 #   Ingest target table with deduplicated data
    merge_qr = f"""
           MERGE WITH SCHEMA EVOLUTION
                INTO {g_ucBronze}.{v_p_srcSchema}.{v_tgt_tbl} t
                USING final s
                ON s.{pk_ls} = t.{pk_ls}
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT * 
    """
    ingest = spark.sql(merge_qr)
    #Summary
    print(f'Merge Completed Successfully')
    print(f'New Columns added : {upward_detected}')   
    print(f'Columns dropped : {downward_detected}')
    print(f'{ingest}')

In [0]:
def refresh_tbl(meta):  
    
    sql=  f"""CREATE OR REPLACE TABLE {v_ucBronze}.{v_p_srcSchema}.{v_tgt_tbl}
                    USING DELTA
                    AS 
                    SELECT  *
                    FROM parquet.`{meta}`"""
    spark.sql(sql)
    if g_env=='DEV':
        spark.sql(f"alter table {v_ucBronze}.{v_p_srcSchema}.{v_tgt_tbl} set owner to {g_ad_group}")       
    print(f"{v_ucBronze}.{v_p_srcSchema}.{v_tgt_tbl} full load was completed.\n")

In [0]:
##### current timestamp in UTC #####
v_ts = spark.sql("select current_timestamp()").collect()[0][0] 

v_maxWorkers = int(dbutils.widgets.get('p_maxWorkers'))

##### if the parameter values does not match Lake House schemas, then fail the notebook #####
v_p_srcSchema = dbutils.widgets.get('p_srcSchema')
if v_p_srcSchema not in [['tcrstat', 'tcrrlcl', 'tcrroo1', 'tcrrhdr']]:
    raise Exception("Source schema was not correctly specified.") ### To fail the notebook if no source schema is provided



In [0]:
v_src_root_path = f"abfss://dw-gqap@{g_01_saLand}/{v_p_srcSchema}/F"

try: 
    v_src_path =   get_sub_folders(v_src_root_path, max_level=3).agg(max("folder").alias("folder")).collect()[0][0]
    v_src_folder_full = get_sub_folders(v_src_path, max_level=1).select("folder").collect()
    display(f"The most recent version is stored in {v_src_path}")

    ##### Convert dataframe to Python list #####
    ls_meta = []
    for i in v_src_folder_full:
        ls_meta.append(i[0])
        print(i[0])
except Exception as e:
    dbutils.notebook.exit('Source folder cannot be found.')    

In [0]:
##### Function to run function refresh_tbl() in parallel #####
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=v_maxWorkers) as executor:
    executor.map(refresh_tbl, ls_meta)